# Recurrent Neural Networks (RNNs)

Goal:

Learn how to model **sequential data**

Examples:

- time series
- molecular sequences
- text (DNA, SMILES, NLP)

Key idea:

Data is not independent --> order matters

## Why Not Use Normal Models?

### Standard ML assumption:

each sample is independent

### But many problems are sequential:

- time series --> past affects future
- text --> previous words matter
- molecules (SMILES) --> order of atoms matters

### Problem

We need a model that remembers:

    past information


## What is an RNN?

### Idea

Process data step-by-step:

    x1 --> x2 --> x3 --> ...

### At each step:

- take current input
- use previous state (memory)
- update internal state

### Structure:

hidden_state_t = f(input_t, hidden_state_t-1)

### Key Insight

RNN =

    model with memory


In [ ]:
# SIMPLE SEQUENCE DATA

import numpy as np

# create simple sequence dataset
# predict next step

def generate_sequence_data(n_samples=1000, seq_length=5):
    
    X = []
    y = []
    
    for _ in range(n_samples):
        seq = np.random.randint(0, 10, size=seq_length)
        
        X.append(seq[:-1])      # input
        y.append(seq[-1])       # predict last value
    
    return np.array(X), np.array(y)

X, y = generate_sequence_data()

print("X shape:", X.shape)
print("y shape:", y.shape)

## What does the dataset represent?

Each row:

    a sequence

Example:

    [2, 5, 7, 8] --> predict next value

### Important

This is a sequence prediction problem

In [ ]:
# TORCH SETUP

import torch
import torch.nn as nn

# convert to tensors
X_torch = torch.tensor(X, dtype=torch.float32)
y_torch = torch.tensor(y, dtype=torch.long)

### 1. The Core Problem

Most ML models assume:

    each sample is independent

But many real-world problems are NOT:

- time series --> past influences future  
- text --> previous words matter  
- molecules (SMILES) --> order defines structure  


### 2. Manual Solution (Intuition)

If you were solving this manually:

You would say:

    "To predict the next value, I need to remember the previous ones."

Example:

Sequence:
    [2 --> 5 --> 7 --> ?]

You mentally keep track of:

    previous values → infer pattern

### Problem:

Manual tracking becomes impossible when:

- sequences are long  
- patterns are complex  

### 3. RNN Solution (Abstraction)

RNN introduces:

    memory (hidden state)

At each step:

    h_t = f(x_t, h_{t-1})

Where:

- x_t = current input  
- h_{t-1} = previous memory  
- h_t = updated memory  

### Key Idea:

    The model carries information through time

### 4. Step-by-Step Processing

For sequence:

    [x1, x2, x3, x4]

The RNN does:

Step 1:
    h1 = f(x1, h0)

Step 2:
    h2 = f(x2, h1)

Step 3:
    h3 = f(x3, h2)

Step 4:
    h4 = f(x4, h3)

At the end:

    use h4 --> predict output

### Important:

The SAME function f is used at every step

--> shared weights

### 5. Internal Mechanism

Each step computes:

    h_t = tanh(Wx * x_t + Wh * h_{t-1} + b)

Meaning:

- combine new input  
- combine memory  
- apply nonlinearity  

### Interpretation:

RNN is continuously updating:

    "what it remembers"

### 6. What is the Hidden State?

Hidden state = compressed memory of past

Example:

Sequence:
    "The cat sat on the ___"

Hidden state contains:

- grammar structure  
- subject ("cat")  
- context  

### Key Insight:

Hidden state is NOT raw data

--> it is a learned representation of history

### 7. Training RNNs

Training uses:

    Backpropagation Through Time (BPTT)

### How it works:

Error at final step depends on:

- all previous steps

So gradients flow:

    backward through sequence


In [ ]:
# RNN MODEL

class SimpleRNN(nn.Module):
    
    def __init__(self, input_size=1, hidden_size=16, output_size=10):
        super().__init__()
        
        # RNN layer
        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True
        )
        
        # output layer
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        
        # reshape input
        x = x.unsqueeze(-1)
        
        # RNN forward
        out, hidden = self.rnn(x)
        
        # use last time step
        out = out[:, -1, :]
        
        # classification
        out = self.fc(out)
        
        return out

model = SimpleRNN()

## Training Process

### Forward pass

compute predictions

### Loss

measure error

### Backpropagation

compute gradients through time

### Optimization

update weights

### Key Insight

RNN learns sequential patterns by adjusting weights

In [ ]:
# TRAINING

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(50):
    
    optimizer.zero_grad()
    
    outputs = model(X_torch)
    
    loss = criterion(outputs, y_torch)
    
    loss.backward()
    
    optimizer.step()
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

In [ ]:
preds = model(X_torch).argmax(dim=1)

accuracy = (preds == y_torch).float().mean()

print("Accuracy:", accuracy.item())


### Key difficulty:

Long sequences --> gradients become:

- very small (vanishing gradient)  
- very large (exploding gradient)  

### Limitations of Basic RNN

### Problem 1 - Vanishing Gradient

Early information gets lost

Example:

    first word becomes irrelevant in long sentence

### Problem 2 - Short Memory

RNN struggles with long-term dependencies

### Problem 3 - Training Instability

Hard to optimize

### 9. Why RNN Still Matters

Even with limitations:

RNN introduced:

    sequence modeling paradigm

Modern models build on this:

- LSTM --> improved memory  
- GRU --> simplified gating  
- Transformers --> attention-based memory  

### 10. Scientific Interpretation

Think in your domain:

Molecule (SMILES):

    "C=O-N-CH3..."

RNN processes:

    one atom at a time

Hidden state represents:

    partial structure of molecule

Prediction could be:

- next atom  
- property  
- classification  

### 11. Key Takeaways

RNN =

    model with memory over sequence

Important properties:

- sequential processing  
- shared weights  
- internal state update  
- temporal dependency learning  

### Final Insight

RNN is NOT just a model

It is a new way of thinking about data:

    data =/= independent samples

    data = evolving sequence
